# Satellite nowcasting: U-Net vs persistence vs optical flow (Kaggle, GPU)

**Settings:** Internet **on**, accelerator **GPU T4**. Add the output of the ingest notebook as input data
(**Add Input → Your Work → notebook output**). Run with **Save & Run All**; results go to `/kaggle/working/runs`.

In [ ]:
!git clone -q --depth 1 https://github.com/ngocquangbiii06/hcmc-storm-nowcasting.git /tmp/stormcast
!pip install -q -e "/tmp/stormcast[ml]"
import os, sys, importlib
os.chdir('/tmp/stormcast'); sys.path.insert(0, '/tmp/stormcast/src')
# fail loudly here rather than 141 times per day later
for mod in ['satpy', 'pyresample', 'xarray', 'zarr', 'boto3'] + ['torch', 'cv2']:
    importlib.import_module(mod)
print(sys.version.split()[0], 'dependencies ok')

In [ ]:
import glob, yaml
zarr = sorted(glob.glob('/kaggle/input/**/himawari_aug2025.zarr', recursive=True))[0]
print('dataset:', zarr)
cfg = yaml.safe_load(open('configs/default.yaml'))
cfg['ml'].update(sat_zarr_path=zarr, output_dir='/kaggle/working/runs', num_workers=2)
yaml.safe_dump(cfg, open('configs/kaggle.yaml', 'w'))

In [ ]:
!python -m stormcast.ml.train_nowcast --config configs/kaggle.yaml --epochs 1 --max-train-batches 10

In [ ]:
!python -m stormcast.ml.train_nowcast --config configs/kaggle.yaml

In [ ]:
import json
from IPython.display import Image, display
run = sorted(glob.glob('/kaggle/working/runs/nowcast-*'))[-1]
display(Image(f'{run}/skill_vs_lead.png'))
test = json.load(open(f'{run}/metrics.json'))['test']
for method, per_lead in test.items():
    print(f'{method:13s}', '  '.join(f"{k}: {v['csi@235K']:.3f}" for k, v in per_lead.items()))